# Tidyverse — Technical Reference

R companion to the Pandas reference. dplyr / tidyr / stringr / lubridate cover the
same ground pandas does — selection, filtering, joins, grouping, window functions,
dates, strings, I/O — with a pipe-based (`|>`) idiom instead of method chaining.

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Data inspection | First look at any data frame | §1 |
| Selection & filtering | Subset rows and columns | §2 |
| NULL (`NA`) handling | Missing values | §3 |
| Data manipulation | Clean, transform, create columns | §4 |
| Joins & merges | Combine data frames | §5 |
| Aggregation & GroupBy | Summarize, reshape, pivot | §6 |
| Window operations | Rank, shift, cumulative, rolling | §7 |
| Date & time | Date arithmetic, extraction | §8 |
| String operations | Text cleaning and extraction | §9 |
| I/O | Read and write data | §10 |
| Performance tips | Speed and memory optimization | §11 |

---
## When to Use

| Signal | tidyverse function to reach for |
| :--- | :--- |
| First look at a data frame | `glimpse()`, `summary()`, `head()` |
| Filter rows by condition | `filter()` |
| Select rows and columns by label/position | `select()` + `filter()` (no combined `.loc` equivalent — see §2) |
| Replace NAs | `tidyr::replace_na()` or `dplyr::coalesce()` |
| Apply condition to create a column | `dplyr::if_else()` (binary) or `dplyr::case_when()` (multi) |
| Summarize with aggregation | `group_by() |> summarise()` |
| Apply aggregation but keep original shape | `group_by() |> mutate()` |
| Rank within a group | `group_by() |> mutate(dense_rank(...))` |
| Compare to previous / next row | `group_by() |> mutate(lag(x))` / `lead(x)` |
| Running total within a group | `group_by() |> mutate(cumsum(x))` |
| Pivot rows to columns | `tidyr::pivot_wider()` |
| Combine two data frames on a key | `*_join()` family |
| Find rows with no match | `anti_join()` |
| Moving average | `slider::slide_dbl(x, mean, .before = n-1)` |
| Parse date strings | `lubridate::ymd()`, `ymd_hms()`, etc. |
| Extract date parts | `lubridate::year()`, `month()`, `wday()` |
| String pattern match | `stringr::str_detect()` |
| Extract regex groups from string | `stringr::str_match()` |

---
## §1 — Data Inspection

Always run these first. They tell you shape, types, missing values, and distributions before you touch the data.

In [ ]:
library(dplyr)
library(tidyr)

# Shape and preview
dim(df)                          # c(rows, cols)
head(df, 5)                      # first 5 rows
tail(df, 5)                      # last 5 rows

# Column types and null counts
glimpse(df)                      # types + a row preview per column -- R's closest analog to df.info()
sapply(df, class)                # just the types

# Summary statistics
summary(df)                      # numeric columns: min, quartiles, mean, max; counts NAs
# summary() already includes character/factor columns (counts + a few values) -- no separate include='all' needed

# Cardinality and distributions
n_distinct(df$col)               # number of distinct values
table(df$col)                    # frequency of each value
prop.table(table(df$col))        # as proportions (0-1)

# NULL summary across all columns
colSums(is.na(df))               # null count per column
colMeans(is.na(df))              # null rate per column

**Common mistakes:**
- Skipping `glimpse()` and assuming types — date columns often load as `character` from `read_csv()` and need explicit parsing with `lubridate`
- Using `summary()` without realizing it silently treats character columns differently from factors — convert intentionally with `as.factor()` if you want level counts
- Confusing `sum(!is.na(df$col))` (excludes NA) with `nrow(df)` (includes all rows)

---
## §2 — Selection & Filtering

`select()` picks columns, `filter()` picks rows by condition, `slice()` picks rows by position. Unlike pandas' combined `.loc[rows, cols]`, tidyverse keeps row and column selection as **separate verbs**, chained with the pipe.

In [ ]:
# Column selection
df |> select(col)                          # single column -> tibble (still a data frame, not a "Series")
df |> select(col1, col2)                   # multiple columns -> tibble
df |> pull(col)                            # single column -> bare vector (closest analog to pandas df['col'])

# Row selection by position (tidyverse's closest analog to .iloc)
df |> slice(1)                             # first row
df |> slice(1:5)                           # first 5 rows (1-indexed, INCLUSIVE end)
df |> slice_tail(n = 1)                    # last row

# Row + column together, by label-like chaining (tidyverse's closest analog to .loc)
df |> filter(mask_condition) |> select(col1, col2)

# Boolean / logical indexing -- expressed via filter(), reads like SQL WHERE
df |> filter(age > 30)                                  # single condition
df |> filter(age > 30 & region == "US")                 # AND -- same & as pandas, but no parens trap (see below)
df |> filter(age < 18 | age > 65)                        # OR
df |> filter(!(status == "inactive"))                    # NOT
df |> filter(platform %in% c("iOS", "Android"))           # IN
df |> filter(!(user_id %in% blocked_ids))                 # NOT IN

# filter() reads as natural R -- no special "query string" syntax needed (this IS the readable form)
df |> filter(age > 30 & region == "US")
threshold <- 100
df |> filter(revenue > threshold)                         # plain variable reference, no @ prefix needed

**`select`/`filter`/`pull` vs pandas' `[]`/`.loc`/`.iloc`:**

| | Rows | Columns | End inclusive? |
| :--- | :--- | :--- | :--- |
| `select()` | n/a (columns only) | by name | n/a |
| `filter()` | by condition | n/a (rows only) | n/a |
| `slice()` | by position | n/a | Yes — `1:5` includes row 5 |
| `pull()` | n/a | single column, returns bare vector | n/a |

R has no single combined "rows-and-columns-by-label" function the way `.loc` is — tidyverse deliberately keeps row-selection (`filter`/`slice`) and column-selection (`select`) as separate, composable verbs chained with the pipe.

**Common mistakes:**
- Reaching for a pandas-style `df[rows, cols]` syntax — base R data frames support `df[rows, cols]` too, but it is rarely used in tidyverse code; prefer `filter() |> select()`
- Forgetting that `filter()` silently drops rows where the condition evaluates to `NA` (neither TRUE nor FALSE) — unlike pandas, where `NaN` comparisons are also falsy but the *reason* differs; always check `is.na()` explicitly if NA rows matter
- Using `select(-col)` to drop a column and forgetting the minus sign applies per-column, not as a vector — use `select(-c(col1, col2))` to drop multiple

---
## §3 — NULL (`NA`) Handling

`NA` is R's null marker — conceptually identical to pandas' `NaN`/`None`, but a single consistent value across all types (numeric, character, logical), unlike pandas' historical `NaN`/`NaT`/`None` split.

In [ ]:
# Detect NAs
is.na(df)                                       # IS NULL -- element-wise logical matrix
!is.na(df)                                      # IS NOT NULL
sum(is.na(df$col))                              # count NAs in a column

# Fill NAs
tidyr::replace_na(df$salary, 0)                 # IFNULL(salary, 0)
df$contact <- dplyr::coalesce(df$phone, df$email, "Unknown")  # COALESCE(phone, email, 'Unknown')
df$col <- tidyr::replace_na(df$col, mean(df$col, na.rm = TRUE))  # fill with column mean

# Forward / backward fill within a group
df <- df |> group_by(user_id) |> tidyr::fill(value, .direction = "down") |> ungroup()  # fill from previous row in group
df <- df |> group_by(user_id) |> tidyr::fill(value, .direction = "up")   |> ungroup()  # fill from next row in group

# Drop rows with NAs
df |> tidyr::drop_na()                          # drop rows with ANY NA
df |> tidyr::drop_na(user_id, date)             # drop rows where specific columns are NA
df[rowSums(!is.na(df)) > 0, ]                   # drop rows where ALL values are NA (no single verb for this)

# NA behavior in aggregations -- R requires EXPLICIT na.rm, pandas ignores NaN by default
sum(df$col, na.rm = TRUE)      # NA must be explicitly excluded -- sum(df$col) alone returns NA if any NA present!
mean(df$col, na.rm = TRUE)     # same -- denominator is non-NA count when na.rm=TRUE
sum(!is.na(df$col))            # excludes NA
nrow(df)                       # includes NA rows

**Common mistakes:**
- Forgetting `na.rm = TRUE` — this is the single biggest pandas->R surprise. `sum(c(1, 2, NA))` returns `NA`, not `3`. Pandas silently skips NaN in aggregations; R does **not**, by design, and requires you to opt in
- Using `df$col == NA` — like pandas, `NA` is never equal to anything, including itself; always use `is.na()`
- `replace_na(df$col, 0)` before computing a mean when NAs should be excluded — changes the denominator and deflates the mean, identical trap to pandas' `fillna(0)` before `.mean()`
- `fill()` without `arrange()`-ing first — fills in row order, not logical time order; always sort before forward filling, same discipline as pandas' `ffill()`

---
## §4 — Data Manipulation

Covers column creation, renaming, type casting, and conditional logic — the day-to-day cleaning toolkit, built around `mutate()`.

In [ ]:
# Add / overwrite columns
df <- df |> mutate(new_col = a + b)            # vectorized arithmetic, chainable by construction

# Rename columns
df |> rename(new = old, b = a)                  # rename specific columns (NOTE: new = old order, reversed from a dict!)
names(df) <- c("col1", "col2", "col3")          # rename all columns at once

# Drop columns or rows
df |> select(-col1, -col2)                      # drop columns
df |> slice(-c(1, 2, 3))                        # drop rows by position

# Type casting
df |> mutate(col = as.integer(col))
df |> mutate(col = as.double(col))
df |> mutate(col = as.character(col))
df |> mutate(col = as.factor(col))              # memory-efficient for low-cardinality strings (R's category dtype)

# Conditional logic
# if_else -- binary (IF/ELSE), and STRICTER than pandas' np.where: both branches must be the same type
df <- df |> mutate(spender_type = if_else(spend > 100, "High", "Low"))

# case_when -- multi-condition (CASE WHEN equivalent), evaluated top to bottom, first match wins
df <- df |> mutate(spender_type = case_when(
  spend > 100              ~ "High",
  spend >= 50 & spend <= 100 ~ "Medium",
  .default = "Low"
))

# cut() -- bin continuous values into labeled ranges (base R, tidyverse has no separate verb)
df <- df |> mutate(age_group = cut(
  age,
  breaks = c(0, 18, 35, 60, 100),
  labels = c("Under 18", "18-35", "36-60", "60+")
))

# recode / case_match -- replace values using a lookup (SQL CASE WHEN on discrete values)
df <- df |> mutate(platform_label = case_match(platform_code,
  1 ~ "iOS", 2 ~ "Android", 3 ~ "Web"
))

# mutate with stringr/custom function -- row-wise or vectorized custom logic (vectorized by default, NOT a Python-style loop)
df <- df |> mutate(col = stringr::str_trim(stringr::str_to_lower(col)))   # acceptable, vectorized
df <- df |> mutate(result = a + b)              # prefer direct vectorized ops over rowwise() when possible

# Deduplication
df |> distinct()                                # drop fully duplicate rows
df |> distinct(user_id, .keep_all = TRUE)       # keep first occurrence per user_id
df |> arrange(desc(row_order)) |> distinct(user_id, .keep_all = TRUE)  # keep "last" -- sort first, R has no keep='last' arg

**Common mistakes:**
- `case_when()` conditions are evaluated top to bottom — first matching condition wins, same rule as pandas' `np.select`; order matters
- `if_else()` is **strict about type** — `if_else(cond, "High", 1)` errors in R, where `np.where(cond, "High", 1)` would silently coerce; this strictness catches real bugs at the cost of occasional friction
- `rename(new = old)` reads backwards from a Python dict's `{'old': 'new'}` — the new name comes first in R's syntax
- Reaching for `rowwise() |> mutate(...)` out of pandas `apply(axis=1)` habit when a vectorized `mutate()` would work — same 10-100x slowdown trap as pandas' row-wise apply

**Choosing a value-substitution / labeling tool:**

| | Selects on | Unmatched values become | SQL analogue |
| :--- | :--- | :--- | :--- |
| `if_else(cond, x, y)` | one logical condition | n/a (binary) | `IF` / `CASE WHEN ... ELSE` |
| `case_when(cond ~ x, ..., .default = y)` | many conditions (first wins) | `.default` | multi-branch `CASE WHEN` |
| `case_match(x, val ~ new, ...)` | discrete value lookup | `NA` unless `.default` set | `CASE` on discrete values |
| `recode(x, old = "new")` | discrete value lookup | **unchanged** (superseded by `case_match`, shown for pandas `.replace()` parity) | targeted `REPLACE` |

The `case_match`-vs-`recode` split mirrors pandas' `map`-vs-`replace` split: unmatched values become `NA` with `case_match` (like `map`) unless you set `.default`, while `recode` leaves them alone (like `replace`).

**`cut()` for binning, no separate `qcut()` in base R:** pandas' `pd.qcut` (equal-frequency / quantile bins) has no single base-R equivalent — combine `quantile()` to compute the breakpoints, then pass them into `cut()`.

In [ ]:
s <- c(10, 20, 30, 40)
cat("if_else(s>20) keeps where TRUE :", if_else(s > 20, s, 0), "\n")
# R has no direct `mask()` (keep-where-FALSE) verb -- flip the condition instead, equally readable:
cat("if_else(!(s>20)) keeps where FALSE:", if_else(!(s > 20), s, 0), "\n")

vals <- c(1, 2, 3, 4, 5, 6, 7, 8, 100)
cut_bins  <- cut(vals, breaks = 3)                                  # equal-width
qcut_bins <- cut(vals, breaks = quantile(vals, probs = seq(0, 1, length.out = 4)), include.lowest = TRUE)  # equal-freq
cat("\ncut  (equal-width) bucket sizes:", as.vector(table(cut_bins)), "\n")
cat("qcut (equal-freq)  bucket sizes:", as.vector(table(qcut_bins)), "\n")

---
## §5 — Joins & Merges

The `*_join()` family (`inner_join`, `left_join`, `right_join`, `full_join`, `anti_join`, `semi_join`) covers everything `.merge()` does — and, unlike pandas, joins **always** match on columns, never silently on the row index, so the `.join()`-vs-`.merge()` footgun simply doesn't exist in dplyr.

In [ ]:
library(dplyr)

# Basic join -- SQL JOIN equivalents
inner_join(A, B, by = "id")           # INNER JOIN
left_join(A, B, by = "id")            # LEFT JOIN
right_join(A, B, by = "id")           # RIGHT JOIN
full_join(A, B, by = "id")            # FULL OUTER JOIN

# Join on multiple keys
left_join(A, B, by = c("user_id", "date"))

# Join on differently named columns
inner_join(A, B, by = c("user_id" = "id"))

# Handle overlapping column names
left_join(A, B, by = "id", suffix = c("_a", "_b"))

# Find rows in A with NO match in B -- LEFT JOIN ... WHERE B.id IS NULL
# dplyr has a PURPOSE-BUILT verb for this -- no merge(indicator=True) + filter dance needed
no_match <- anti_join(A, B, by = "id")

# Self join -- join a data frame to itself
left_join(
  employees,
  employees |> select(id, name) |> rename(manager_id = id, manager_name = name),
  by = "manager_id"
)

# Stack data frames vertically -- SQL UNION ALL
bind_rows(df1, df2)                    # UNION ALL (always resets row names, pandas' ignore_index=True is the default behavior here)
bind_rows(df1, df2) |> distinct()      # UNION (distinct)

**Common mistakes:**
- Joining on columns with different types (e.g. `integer` vs `character`) — `*_join()` raises a type-mismatch error rather than silently producing zero matches the way pandas' `.merge()` can; this is actually a safety improvement, but it means you must cast explicitly *before* R will let you proceed
- Forgetting `suffix =` when both data frames have overlapping column names — dplyr adds `.x` / `.y` by default (pandas adds `_x` / `_y`, same idea, different characters)
- Out of pandas habit, reaching for base R's index-aligning `merge()` defaults incorrectly — dplyr's `*_join()` functions **always** match on the `by=` columns, never on row position/index, so there is no `.join()`-vs-`.merge()` ambiguity to navigate in tidyverse code
- `bind_rows()` with mismatched column sets — unlike pandas' `pd.concat` (which fills missing columns with NaN automatically), `bind_rows()` does the same thing, but only for columns that exist in at least one frame; entirely new column names introduced in just one frame still need to make sense semantically

**`*_join()` vs base R `merge()`, and `bind_rows` vs `*_join`:**

| | Aligns on | Default keys | Prefer when |
| :--- | :--- | :--- | :--- |
| `dplyr::*_join(A, B, by = "id")` | **columns**, explicit | none — you specify `by` | Almost always — explicit, SQL-like, handles renamed keys |
| `base::merge(A, B)` | columns (auto-detects shared names) | shared column names | Legacy code, or no dplyr dependency desired |

dplyr's `*_join()` family never has a "wrong index alignment" failure mode the way pandas' `.join()` does, because there is no index-based join verb in normal tidyverse usage at all.

| | Combines by | Use for |
| :--- | :--- | :--- |
| `bind_rows(A, B)` | stacking rows (or `bind_cols` for columns) — no key matching | `UNION ALL` / gluing partitions |
| `*_join(A, B, by = k)` | matching a key | `JOIN` |

In [ ]:
A <- tibble::tibble(id = c(1, 2, 3), val = c("a", "b", "c"))
B <- tibble::tibble(id = c(2, 3, 4), score = c(20, 30, 40))

cat("inner_join on 'id' -- matches on the key:\n")
print(inner_join(A, B, by = "id"))

cat("\nanti_join -- rows in A with NO match in B (no separate indicator step needed):\n")
print(anti_join(A, B, by = "id"))

---
## §6 — Aggregation & GroupBy

Three distinct verbs with different output shapes: `summarise()` collapses rows, `mutate()` after `group_by()` preserves shape, `pivot_wider()` reshapes across two dimensions.

In [ ]:
# SQL pipeline equivalent
# SELECT col1, AGG(col2) FROM t WHERE cond GROUP BY col1 HAVING ... ORDER BY ... LIMIT n
result <- df |>
  filter(condition) |>                              # WHERE
  group_by(col1) |>                                 # GROUP BY
  summarise(agg_col = sum(col2), .groups = "drop") |>  # SELECT + aggregate (ungrouped result by default)
  filter(agg_col > 100) |>                          # HAVING
  arrange(desc(col1)) |>                            # ORDER BY
  slice_head(n = 10)                                # LIMIT

# Basic aggregations
nrow(df)                                # COUNT(*)
sum(!is.na(df$col))                     # COUNT(col) -- excludes NA
n_distinct(df$user_id)                  # COUNT(DISTINCT user_id)
sum(df$sales, na.rm = TRUE)             # SUM
mean(df$price, na.rm = TRUE)            # AVG

# Named aggregation in group_by -- this IS the preferred dplyr syntax, no separate "named" variant needed
result <- df |> group_by(region) |> summarise(
  total_sales  = sum(sales, na.rm = TRUE),
  avg_price    = mean(price, na.rm = TRUE),
  unique_users = n_distinct(user_id),
  min_score    = min(score, na.rm = TRUE),
  .groups = "drop"
)

# Conditional aggregation -- SUM(CASE WHEN ...) equivalent
df <- df |> mutate(
  ios_rev      = if_else(platform == "iOS", revenue, 0),       # else 0
  ios_rev_excl = if_else(platform == "iOS", revenue, NA_real_) # else NA (excluded from mean)
)
result <- df |> group_by(region) |> summarise(
  ios_total = sum(ios_rev, na.rm = TRUE),
  ios_avg   = mean(ios_rev_excl, na.rm = TRUE),
  .groups = "drop"
)

# mutate after group_by -- aggregation that keeps original shape (window function equivalent)
df <- df |> group_by(region) |> mutate(
  group_total = sum(revenue, na.rm = TRUE),
  pct_of_group = revenue / group_total
) |> ungroup()

# pivot_wider -- GROUP BY across two categorical dimensions
df |>
  group_by(region, platform) |>
  summarise(revenue = sum(revenue, na.rm = TRUE), .groups = "drop") |>
  pivot_wider(names_from = platform, values_from = revenue, values_fill = 0)

# table() -- frequency count only (pivot_wider shortcut, R's crosstab equivalent)
table(df$region, df$platform)

**`summarise` vs `group_by |> mutate` vs `pivot_wider`:**

| | Output shape | SQL equivalent | Use when |
| :--- | :--- | :--- | :--- |
| `summarise()` | Collapsed (one row per group) | `GROUP BY` | You want a summary table |
| `group_by() |> mutate()` | Same as input | Window function | You need the result back on every row |
| `pivot_wider()` | Reshaped (groups become columns) | `GROUP BY` + conditional agg | Two categorical dimensions |

**Common mistakes:**
- Forgetting `.groups = "drop"` in `summarise()` — leaves the result grouped (by the remaining group levels), which silently changes behavior of every verb chained afterward; this is dplyr's analog to forgetting `as_index=False`
- Forgetting `ungroup()` after a `group_by() |> mutate()` block — the grouping persists and corrupts later operations exactly like a forgotten `.groups` arg
- `pivot_wider()` without `values_fill = 0` — missing combinations produce `NA` instead of 0, same trap as pandas' `pivot_table` without `fill_value=0`
- Summarising multiple statistics with `across(everything(), list(...))` and not naming them — produces auto-generated `col_fn1` style names, similar confusion to pandas' `.agg(['sum','mean'])` MultiIndex trap; prefer explicit named `summarise()` arguments as shown above

**Reshaping verbs, and three "count" methods:**

| | Aggregates duplicates? | Errors on dup keys? |
| :--- | :--- | :--- |
| `tidyr::pivot_wider(...)` | No by default — errors/lists if duplicate index/column pairs exist unless `values_fn` is given | Without `values_fn`, warns and produces list-columns |
| `tidyr::pivot_wider(..., values_fn = sum)` | Yes, via `values_fn` | No |

| Method | Counts | SQL equivalent |
| :--- | :--- | :--- |
| `table(df$col)` | rows per distinct **value** of one column | `GROUP BY col` + `COUNT(*)` |
| `df |> count(k)` | rows per **group** (dplyr's `groupby().size()` equivalent) | `GROUP BY k` + `COUNT(*)` |
| `n_distinct(df$col)` | number of **distinct** values | `COUNT(DISTINCT col)` |

**`distinct()` vs `group_by() |> slice(1)`:**

| | Keeps | Other columns |
| :--- | :--- | :--- |
| `df |> distinct(k, .keep_all = TRUE)` | first row per key | kept as-is from that row |
| `df |> group_by(k) |> summarise(across(everything(), ~first(na.omit(.))))` | first non-null per column per group | can differ column-by-column; lets you aggregate the rest |

---
## §7 — Window Operations

Covers ranking, lag/lead, running totals, and rolling windows — the dplyr equivalents of SQL window functions, all expressed as `group_by() |> mutate(window_fn(...))`.

In [ ]:
# Ranking -- SQL RANK / DENSE_RANK / ROW_NUMBER equivalents
df <- df |> mutate(
  row_num   = row_number(desc(score)),    # ROW_NUMBER
  rnk       = min_rank(desc(score)),      # RANK
  dense_rnk = dense_rank(desc(score))     # DENSE_RANK
)

# Ranking within a group -- RANK() OVER (PARTITION BY ...)
df <- df |> group_by(region) |> mutate(
  rank_in_group = dense_rank(desc(revenue))
) |> ungroup()

# LAG / LEAD -- shift within a group (always arrange first)
df <- df |> arrange(user_id, date) |> group_by(user_id) |> mutate(
  prev_value = lag(value, 1),    # LAG(value, 1)
  next_value = lead(value, 1)    # LEAD(value, 1)
) |> ungroup()

# Running totals -- SUM OVER (ROWS UNBOUNDED PRECEDING)
df <- df |> group_by(user_id) |> mutate(
  run_sum = cumsum(amount),
  run_max = cummax(amount),
  run_min = cummin(amount)
) |> ungroup()

# Rolling window -- moving average over last n rows (needs the slider package; base R has no rolling())
df <- df |> group_by(user_id) |> mutate(
  moving_avg_7 = slider::slide_dbl(amount, mean, .before = 6, .complete = FALSE)
) |> ungroup()

# Expanding window -- cumulative from start of series
df <- df |> group_by(user_id) |> mutate(
  expanding_mean = slider::slide_dbl(amount, mean, .before = Inf)
) |> ungroup()

**Rank function comparison:**

| function | Ties | Gaps after ties | SQL equivalent |
| :--- | :--- | :--- | :--- |
| `row_number()` | No — sequential, ties broken by row order | n/a | `ROW_NUMBER()` |
| `min_rank()` | Yes — share min rank | Yes | `RANK()` |
| `dense_rank()` | Yes — share rank | No | `DENSE_RANK()` |

**Common mistakes:**
- Forgetting to `arrange()` before `lag()`/`lead()` — operates on current row order within the group, not logical order, same trap as pandas' unsorted `shift()`
- Forgetting `ungroup()` after a windowed `group_by() |> mutate()` — the grouping silently persists into later pipe steps
- Reaching for base R's `zoo::rollmean()` out of old habit instead of `slider::slide_dbl()` — `slider` integrates more cleanly with `dplyr`'s grouped `mutate()` and handles partial windows (`.complete=`) explicitly
- `rank()` (base R, not `dplyr::min_rank()`) handles ties differently by default (`"average"`) — when porting pandas' `rank(method=...)`, reach for dplyr's `row_number`/`min_rank`/`dense_rank` trio specifically, not base `rank()`

---
## §8 — Date & Time

`lubridate` is the tidyverse date/time package — its parsing functions (`ymd()`, `mdy()`, `ymd_hms()`, ...) replace pandas' single `pd.to_datetime()` with format-specific functions named after the format itself.

In [ ]:
library(lubridate)

# Parse strings to datetime -- lubridate names the function after the expected format
df <- df |> mutate(event_ts = ymd_hms(event_ts))         # explicit format via function choice (e.g. "2024-01-05 10:30:00")
df <- df |> mutate(event_ts = parse_date_time(event_ts, orders = c("ymd", "mdy", "dmy")))  # auto-detect across formats

# Unix timestamp conversion
df <- df |> mutate(dt = as_datetime(ts_col))              # seconds since epoch -> datetime (lubridate default unit)
df <- df |> mutate(dt = as_datetime(ts_col / 1000))       # milliseconds -> seconds first, then convert

# Extract components
df <- df |> mutate(
  date        = as_date(event_ts),                       # DATE()
  year        = year(event_ts),                          # EXTRACT(YEAR)
  month       = month(event_ts),                          # EXTRACT(MONTH)
  day         = day(event_ts),
  day_of_week = wday(event_ts, week_start = 1) - 1,       # 0=Mon, 6=Sun -- wday() is 1-indexed and Sunday-first by default
  hour        = hour(event_ts),
  year_month  = format(event_ts, "%Y-%m")                 # DATE_FORMAT('%Y-%m')
)

# Date arithmetic
df <- df |> mutate(
  plus_7d   = order_date + days(7),                       # DATE_ADD(..., INTERVAL 7 DAY)
  minus_1mo = order_date %m-% months(1),                  # DATE_SUB(..., INTERVAL 1 MONTH) -- %m-% is calendar-aware subtraction
  days_diff = as.numeric(end_date - start_date, units = "days")  # DATEDIFF
)

# Current date
today <- floor_date(now(), "day")                          # CURRENT_DATE
df_recent <- df |> filter(event_date >= today - days(7))

# Gaps & Islands -- consecutive streak detection
logins <- logins |> arrange(user_id, login_date) |> group_by(user_id) |> mutate(
  row_num = row_number(),
  grp     = login_date - days(row_num)
) |> ungroup()
streaks <- logins |> group_by(user_id, grp) |> summarise(
  streak_start = min(login_date),
  streak_end   = max(login_date),
  streak_len   = n(),
  .groups = "drop"
)

**`lubridate::days()`/`weeks()` (fixed) vs `months()`/`years()` used with `%m+%`/`%m-%` (calendar-aware):**

| | Fixed duration | Calendar-aware | Use for |
| :--- | :--- | :--- | :--- |
| `days(7)`, `weeks(1)` (`Period`/`Duration` class) | Yes | No | Days, hours, minutes |
| `months(1)`, `years(1)` used with plain `+`/`-` | No | Partially — can land on invalid dates (e.g. Jan 31 + 1 month) | Months, years where overflow is acceptable |
| `months(1)` used with `%m+%` / `%m-%` | No | Yes — rolls back to the last valid day of the month instead of overflowing | Months, years where calendar correctness matters most |

**Common mistakes:**
- Forgetting to parse with a `lubridate` function before using date extraction functions — `year()` on a character column raises an error or silently returns `NA`, same failure mode as pandas' `.dt` accessor on an unparsed column
- Using plain `+ months(1)` instead of `%m+%` for month arithmetic — plain arithmetic on Jan 31 + 1 month overflows into March instead of landing on Feb 28/29; `%m+%` is lubridate's deliberate fix, with no direct pandas equivalent since `pd.DateOffset` handles this internally
- `wday()`'s default is **1 = Sunday**, not 0 = Monday like pandas' `dayofweek` — always pass `week_start = 1` and subtract 1 to match the pandas convention exactly, as shown above

---
## §9 — String Operations

All string functions live in `stringr`, prefixed `str_*`. Like pandas' `.str` accessor, they are vectorized and `NA`-safe by default — `NA` rows return `NA` rather than raising an error.

In [ ]:
library(stringr)

# Case and whitespace
df |> mutate(col = str_to_lower(col))
df |> mutate(col = str_to_upper(col))
df |> mutate(col = str_trim(col))                          # remove leading/trailing whitespace
df |> mutate(col = str_to_lower(str_trim(col)))             # chain operations (function composition, not method chaining)

# Contains / starts with / ends with
df |> filter(str_detect(col, "pattern"))                    # LIKE '%pattern%'
df |> filter(str_detect(col, "pattern") & !is.na(col))      # NA rows already return NA from str_detect -- filter() drops NA rows by default, so this is often redundant; shown for parity with pandas' explicit na=False
df |> filter(str_starts(col, "prefix"))
df |> filter(str_ends(col, "suffix"))

# Replace
df |> mutate(col = str_replace(col, "old", "new"))          # REPLACE(col, 'old', 'new') -- first match only
df |> mutate(col = str_replace_all(col, "old", "new"))      # replace ALL occurrences (pandas' str.replace default)
df |> mutate(col = str_remove_all(col, "\\d+"))              # remove all digits (regex is ALWAYS on in stringr -- no regex=True flag needed)

# Split
df |> mutate(col = str_split(col, ","))                     # returns a list-column, same idea as pandas' list-per-cell
df |> separate_wider_delim(col, delim = ",", names_sep = "_")  # expand into separate columns
df <- df |> separate_wider_delim(name, delim = " ", names = c("first", "last"), too_many = "merge")  # split into named cols

# Extract -- pull out groups using regex
df |> mutate(year = str_extract(col, "\\d{4}"))                                  # extract first 4-digit number
df |> mutate(matches = str_match(col, "(?<year>\\d{4})-(?<month>\\d{2})"))        # named groups -> matrix of named columns

# Length
df |> mutate(len = str_length(col))                          # character count per cell

# Slice
df |> mutate(first3 = str_sub(col, 1, 3))                    # first 3 characters (1-indexed, INCLUSIVE end)
df |> mutate(mid = str_sub(col, 3, 5))                       # characters 3-5

**Common mistakes:**
- Assuming `str_detect()` returns `FALSE` for `NA` input the way pandas' `na=False` makes explicit — `str_detect()` actually returns `NA` for `NA` input by default, and `filter()` drops `NA` rows automatically (treats them as not-matching), which usually produces the *same end result* as pandas' `na=False` but for a different mechanical reason — don't assume the equivalence holds outside of `filter()` (e.g. inside `mutate()` an `NA` stays `NA`, it does not become `FALSE`)
- Forgetting that `stringr` regex is **always on** — there is no `regex = TRUE/FALSE` toggle like pandas; use `str_replace()`/`str_detect()` for regex, or `fixed()`/`coll()` wrappers around the pattern for literal matching when the pattern contains regex special characters (`.`, `+`, `*`)
- `str_replace()` only replaces the **first** match per string — reach for `str_replace_all()` for pandas' default all-occurrences behavior, the reverse of pandas where `.str.replace()` replaces all by default

---
## §10 — I/O

`readr` covers reading and writing delimited data; setting column types on load avoids silent type errors downstream, same discipline as pandas' `dtype=` argument.

In [ ]:
library(readr)

# Read CSV
df <- read_csv("file.csv")
df <- read_csv("file.csv", col_select = c(user_id, date, revenue))           # load only needed columns
df <- read_csv("file.csv", col_types = cols(user_id = col_character(), amount = col_double()))  # specify types on load
df <- read_csv("file.csv", col_types = cols(event_date = col_date()))         # parse dates on load
df <- read_csv("file.csv", n_max = 1000)                                      # load first N rows only

# Process large files in chunks
read_csv_chunked("large.csv", callback = DataFrameCallback$new(function(chunk, pos) {
  chunk |> filter(revenue > 0)
}), chunk_size = 100000)

# Read JSON
df <- jsonlite::fromJSON("file.json")
df <- jsonlite::stream_in(file("file.json"))                                  # JSON Lines format

# Read Excel
df <- readxl::read_excel("file.xlsx", sheet = "Sheet1")

# Read SQL
library(DBI)
conn <- dbConnect(RSQLite::SQLite(), "db.sqlite")
df <- dbGetQuery(conn, "SELECT * FROM table WHERE date > '2024-01-01'")

# Write
write_csv(df, "output.csv")                                                   # no row-names column by default (unlike write.csv())
jsonlite::write_json(df, "output.json")
writexl::write_xlsx(df, "output.xlsx")

**Common mistakes:**
- Using base R's `write.csv()` instead of readr's `write_csv()` — `write.csv()` writes row names as an unnamed first column by default (R's analog to forgetting pandas' `index=False`); `write_csv()` does not have this problem
- Loading all columns when only a few are needed — use `col_select` to reduce memory, same idea as pandas' `usecols`
- Not specifying `col_types` for ID columns — numeric IDs load as `double` and lose leading zeros (e.g. zip codes), identical trap to pandas loading them as `int64`

---
## §11 — Performance Tips

In [ ]:
# Prefer vectorized operations over rowwise()/sapply()
df |> mutate(col = col * 2)                       # fast -- vectorized
df |> rowwise() |> mutate(col = col * 2) |> ungroup()   # slow & unnecessary here -- row-by-row, no vectorized op to lose

# filter() vs base R subsetting -- both are fine; filter() is the readable, tidyverse-idiomatic default
df |> filter(age > 30 & region == "US")           # readable, standard tidyverse style
df[df$age > 30 & df$region == "US", ]             # base R equivalent, no real speed difference for in-memory frames

# Reduce memory with factor type (low-cardinality string columns, R's "category dtype")
df <- df |> mutate(platform = as.factor(platform))

# Read only needed columns
read_csv("file.csv", col_select = c(user_id, date, revenue))

# data.table / dtplyr backends -- for genuinely large data, swap the execution engine without rewriting the verbs
# dtplyr::lazy_dt(df) |> filter(...) |> as_tibble()    # dplyr syntax, data.table speed

# Process large files in chunks
read_csv_chunked("large.csv",
  callback = DataFrameCallback$new(function(chunk, pos) process(chunk)),
  chunk_size = 100000)

- Avoid `rowwise()` for anything a vectorized `mutate()` can express — it is R's equivalent of pandas' `apply(axis=1)`, a genuine per-row loop under the hood
- R has no `inplace=True` concept to avoid — every dplyr verb returns a new object by design (consistent with copy-on-modify throughout this stack), so there is nothing to opt out of
- For low-cardinality character columns (platform, country, status), `as.factor()` can reduce memory similarly to pandas' `.astype('category')`
- For genuinely large data that doesn't fit comfortably in memory, `dtplyr` (data.table backend, same dplyr verbs) or `arrow`/`duckdb` (out-of-memory backends with dplyr translation) are the standard escalation path — analogous to reaching for `polars` or chunked pandas processing

---
# Decision Guide

```
Aggregating data?
+-- One metric, one grouping dimension        -> group_by() |> summarise()
+-- One metric across two categorical dims    -> group_by() |> summarise() |> pivot_wider()
+-- Frequency count only                      -> table() / count()
+-- Need result back on every original row    -> group_by() |> mutate()

Filtering rows?
+-- Any condition, readable                   -> filter() -- this IS the readable form, no separate "query" syntax
+-- Dynamic / programmatic condition          -> filter() with a variable reference (no @ prefix needed)

Selecting rows + columns together?
+-- Rows by condition, then columns           -> filter() |> select()
+-- Rows by position, then columns            -> slice() |> select()

Labeling / bucketing values?
+-- Binary condition                          -> if_else(cond, true_val, false_val)
+-- Multiple conditions                       -> case_when(cond1 ~ val1, ..., .default = val)
+-- Binning continuous values                 -> cut(x, breaks = ...) (manual quantile breaks for qcut-style)

Combining data frames?
+-- Join on a column                          -> inner_join() / left_join() / etc.
+-- Rows with no match in other table         -> anti_join()
+-- Stack rows vertically                     -> bind_rows(df1, df2)

Window operations?
+-- Rank within group                         -> group_by() |> mutate(dense_rank(desc(x)))
+-- Previous / next row value                 -> group_by() |> mutate(lag(x)) / lead(x)
+-- Running total                             -> group_by() |> mutate(cumsum(x))
+-- Moving average over last n rows           -> group_by() |> mutate(slider::slide_dbl(x, mean, .before = n-1))

NULL handling?
+-- Replace with a value                      -> replace_na(x, value) / coalesce(...)
+-- Replace with previous row in group        -> group_by() |> fill(x, .direction = "down")
+-- Drop rows                                 -> drop_na(col1, col2, ...)
+-- Aggregations: remember na.rm=TRUE!         -> sum(x, na.rm = TRUE), mean(x, na.rm = TRUE), etc.

Speeding up slow code?
+-- Using rowwise() row-wise                  -> replace with vectorized mutate()
+-- Large file loading slowly                 -> col_select= or read_csv_chunked()
+-- High-cardinality string columns           -> as.factor()
+-- Data too big for memory                   -> dtplyr / arrow / duckdb backends
```